# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [2]:
DIR='../outputs.cds/singleclust/bam'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth_txt(metag, species, *, exclude_ends=75):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo')).select(['gene', 'pos', 'cov'])

    sum_df = df.group_by('gene').all().with_columns(
        # select slice [75:-75]
        (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species")),
        # summarize: average depth across contig,
        (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
        # average depth across covered bases,
        (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
        # fraction of bases covered
        (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
    ).select(["metag", "species", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    return sum_df

read_depth_txt('ERR1135199', 's__Cryptobacteroides sp900546925')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""HLFEEHJE_00326""",1059,934,0.881964,1.807365,2.049251
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""JPFIBGDK_00461""",4623,2261,0.489076,0.684404,1.399381
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_01544""",423,193,0.456265,0.650118,1.42487
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""JPFIBGDK_01534""",681,0,0.0,0.0,NaN
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""EPLMDCHM_00231""",2037,1536,0.75405,1.232204,1.634115
…,…,…,…,…,…,…,…
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""BEKEPKDC_00812""",948,500,0.527426,0.733122,1.39
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""CMHHCIHA_00622""",1185,1012,0.854008,4.637975,5.43083
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_02652""",1374,1165,0.847889,1.47016,1.733906


In [3]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth_txt(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 2000
100 of 2000
200 of 2000
300 of 2000
400 of 2000
500 of 2000
600 of 2000
700 of 2000
800 of 2000
900 of 2000
1000 of 2000
1100 of 2000
1200 of 2000
1300 of 2000
1400 of 2000
1500 of 2000
1600 of 2000
1700 of 2000
1800 of 2000
1900 of 2000
read 2000 depth files.


In [4]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""SRR14369134""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,17800.038462,17800.038462
"""SRR11489750""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,12280.894587,12280.894587
"""SRR12795790""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,8948.836182,8948.836182
"""SRR10209683""","""s__Mogibacterium_A kristiansen…","""AAOFCIJB_00277""",1119,1119,1.0,8170.679178,8170.679178
"""SRR17241663""","""s__Mogibacterium_A kristiansen…","""JMKEGFFJ_01326""",1899,1899,1.0,7469.317536,7469.317536
…,…,…,…,…,…,…,…
"""ERR3211876""","""s__Gemmiger qucibialis""","""PMKFNJDI_00477""",3072,4,0.001302,0.001302,1.0
"""SRR8655118""","""s__Cryptobacteroides sp9005469…","""CMHHCIHA_00893""",2016,2,0.000992,0.000992,1.0
"""SRR11124687""","""s__Prevotella sp002251295""","""OPHKNFIJ_00863""",2094,2,0.000955,0.000955,1.0


## Summarize our mapping breadth results across all the metagenomes

In [5]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [6]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

species,gene,f
str,str,f64
"""s__Cryptobacteroides sp0004326…","""PLMAOHLG_00421""",0.59
"""s__Colivicinus sp002299675""","""OKBMHBJH_00928""",0.62
"""s__Sodaliphilus sp004557565""","""MOGKDPAH_00816""",0.91
"""s__Sodaliphilus sp004557565""","""DAJNDEIP_01218""",0.93
"""s__Sodaliphilus sp004557565""","""ADJFOMCG_01030""",0.84
…,…,…
"""s__Prevotella sp000434975""","""AEONIABF_01853""",0.77
"""s__Prevotella sp000434975""","""PAPCDFOC_01977""",0.85
"""s__Gemmiger qucibialis""","""OHFELPLO_01144""",0.2


In [11]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', '0', species, name, "(not reviewed)"])
    outfp.close()
            
    

s__Bariatricus sp004560705
shape: (14, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.9  │
│ s__Bariatricus sp004560705 ┆ FMMMFAAO_00746 ┆ 0.89 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ IEPLHDOE_02248 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ NOACAEKI_00860 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ NGBMPMHL_00028 ┆ 0.81 │
└────────────────────────────┴────────────────┴──────┘
../outputs.cds/singlecl

## Export mapping abundance data to a CSV, after merging with `species-genes.csv`

In [24]:
species_genes_df = (pl.read_csv('../outputs.cds/singleclust/species-genes.csv')
    .filter(pl.col("good") == 1)
    .with_columns(pl.col('gene_name').alias('gene'))
    .select(["anchor", "gene", "species", "description"])
)

species_genes_df

anchor,gene,species,description
i64,str,str,str
1,"""CNENGHLA_01260""","""s__Phascolarctobacterium_A suc…","""BLAST match to hydrogenase lar…"
0,"""EHOAPHDI_01174""","""s__Phascolarctobacterium_A suc…","""BLAST match to protein phospha…"
0,"""CNENGHLA_00658""","""s__Phascolarctobacterium_A suc…","""BLAST match to 4-hydroxy-3-met…"
0,"""IFIBFMPA_00800""","""s__Phascolarctobacterium_A suc…","""BLAST match to 2-isopropylmal…"
0,"""BBOFCOCJ_01349""","""s__Lactobacillus amylovorus""","""BLAST match to peptidase T [La…"
…,…,…,…
1,"""JBPBJODD_00501""","""s__Cryptobacteroides sp0004349…","""hypothetical protein [Bacteroi…"
0,"""KHMCBGLI_01458""","""s__Cryptobacteroides sp0004349…","""putative uncharacterized prote…"
0,"""FCHBMNJF_01297""","""s__Cryptobacteroides sp0004349…","""ATP-binding protein [Bacteroid…"


In [25]:
merge_df = depth_df.join(species_genes_df, on=["species", "gene"], how='inner')
merge_df

metag,species,gene,len,hits,breadth,depth_all,depth_cov,anchor,description
str,str,str,u32,u32,f64,f64,f64,i64,str
"""SRR17241485""","""s__Phascolarctobacterium_A suc…","""CNENGHLA_00658""",1179,383,0.324852,3.201866,9.856397,0,"""BLAST match to 4-hydroxy-3-met…"
"""SRR17241485""","""s__Phascolarctobacterium_A suc…","""CNENGHLA_01260""",1398,545,0.389843,3.907725,10.023853,1,"""BLAST match to hydrogenase lar…"
"""ERR3211974""","""s__UBA2868 sp004552595""","""IPIMNMMB_00240""",1995,151,0.075689,0.075689,1.0,0,"""type I DNA topoisomerase [Lach…"
"""ERR3211974""","""s__UBA2868 sp004552595""","""IEHIAHIF_00466""",1431,76,0.05311,0.05311,1.0,0,"""L-arabinose isomerase [Lachnos…"
"""ERR3211974""","""s__UBA2868 sp004552595""","""KMEHCMKH_00746""",1083,314,0.289935,0.43675,1.506369,1,"""multiple monosaccharide ABC tr…"
…,…,…,…,…,…,…,…,…,…
"""ERR1135304""","""s__Prevotella sp002251295""","""GFALEAHI_01277""",2526,2516,0.996041,44.572842,44.75,1,"""YfnO"""
"""ERR1135304""","""s__Prevotella sp002251295""","""MNKNIMJG_02022""",186,186,1.0,34.532258,34.532258,0,"""hypothetical"""
"""ERR1135304""","""s__Prevotella sp002251295""","""KINAFDOL_01550""",2604,2180,0.837174,32.506144,38.82844,0,"""transcription-repair coupling …"


In [26]:
merge_df.write_csv('../outputs.cds/cds3-genes/mapping-coverage.csv')